# Geospatial Analysis with DuckDB and Mapbox

In this notebook we'll answer a simple question: **which DC elementary schools are the most walkable?**

We'll measure walkability as the number of residential addresses within a 10-minute walk of each school, using:
- **DuckDB** (with its spatial extension) — a fast analytical database that understands geometry
- **Mapbox Isochrone API** — generates polygons representing areas reachable within a travel-time threshold
- **Mapbox GL JS** — renders an interactive map of the results

## 1. Dependencies

When you ran `uv sync`, it installed everything we need:
- `duckdb` — our geospatial database
- `requests` — for HTTP calls to the Mapbox API
- `jupyter` — to run this notebook

No `pip install` needed.

**You will need a Mapbox access token to use the Mapbox GL JS visualizations and the Isochrone API. Get a free access token at [account.mapbox.com](https://account.mapbox.com). **Replace the placeholder below with your token.**

In [ ]:
MAPBOX_ACCESS_TOKEN=""

## 2. Download the Data

We need two datasets. Create a `data/` folder at the root of this repo and save the files there.

### DC Public Schools
1. Go to [opendata.dc.gov/datasets/dc-public-schools](https://opendata.dc.gov/datasets/dc-public-schools/about)
2. Click **Download** → select **GeoJSON**
3. Save as `data/DC_Public_Schools.geojson`

### DC Address Points
1. Go to [opendata.dc.gov/pages/addressing-in-dc#data](https://opendata.dc.gov/pages/addressing-in-dc#data)
2. Download the **Address Points** shapefile
3. Unzip into `data/Address_Points/` so you have `data/Address_Points/Address_Points.shp` (and `.dbf`, `.shx`, etc.)

Your `data/` folder should look like:
```
data/
  DC_Public_Schools.geojson
  Address_Points/
    Address_Points.shp
    Address_Points.dbf
    Address_Points.prj
    Address_Points.shx
```

Examine the files in QGIS to ensure we're looking at the right thing. Do we understand how to filter the dataset? You may need to go looking for [additional documentation](https://opendata.dc.gov/documents/130778ae88bb433cb0024298c478ab46/explore).

## 4. Load Data into DuckDB

DuckDB's spatial extension can read GeoJSON and shapefiles directly with `ST_Read`. We'll load both datasets into a local database file.

I'll be honest: I don't love including the `ST_FlipCoordinates()` or `ST_Transform()` (or that DuckDB `config` parameter) in this example. The DC datasets have some weirdness baked in that makes it necessary. This is a problem I worked through with Claude, and you shouldn't be surprised if you have to go through similar debugging when onboarding a new dataset.

In [ ]:
import duckdb

conn = duckdb.connect("dc_schools.duckdb", config={"storage_compatibility_version": "v1.5.0"})

conn.execute("INSTALL spatial")
conn.execute("LOAD spatial")

conn.execute("""
    CREATE OR REPLACE TABLE schools AS
    SELECT * FROM ST_Read('data/DC_Public_Schools.geojson')
""")

# The address shapefile is in EPSG:3857 (Web Mercator).
# ST_Transform to EPSG:4326 returns (lat, lon) axis order per the EPSG spec,
# so we flip to (lon, lat) for compatibility with GeoJSON and Mapbox.
conn.execute("""
    CREATE OR REPLACE TABLE addresses AS
    SELECT * EXCLUDE (geom),
           ST_FlipCoordinates(ST_Transform(geom, 'EPSG:3857', 'EPSG:4326')) AS geom
    FROM ST_Read('data/Address_Points/Address_Points.shp')
""")

schools_count = conn.execute("SELECT COUNT(*) FROM schools").fetchone()[0]
addresses_count = conn.execute("SELECT COUNT(*) FROM addresses").fetchone()[0]
print(f"Loaded {schools_count} schools and {addresses_count} addresses")

In [ ]:
print("=== schools ===")
for row in conn.execute("DESCRIBE schools").fetchall():
    print(row)

print("\n=== addresses ===")
for row in conn.execute("DESCRIBE addresses").fetchall():
    print(row)

In [ ]:
print("=== first 3 schools ===")
for row in conn.execute("SELECT * FROM schools LIMIT 3").fetchall():
    print(row)

print("\n=== distinct school levels ===")
for row in conn.execute("SELECT DISTINCT LEVEL_ FROM schools ORDER BY LEVEL_").fetchall():
    print(row[0])

### 3. Visualize 10,000 random address points
The full dataset is hundreds of thousands of points--a lot to stick in browser memory all at once. We'll sample 5,000 random ones from the dataset, and we'll do it twice: once, filtered to everything *NOT* marked non-residential; and once filtered to everything else. This double-negative is a bit confusing! But after looking at the data dictionary, it seems to be the safest way to ensure we're not excluding residential addresses.

In [ ]:
import json
import base64
from IPython.display import IFrame

# extract some residential records
sample_residential = conn.execute("""
    SELECT ST_X(geom) AS lng, ST_Y(geom) AS lat
    FROM addresses
    WHERE RESIDENTIA!='NON RESIDENTIAL'
    ORDER BY RANDOM()
    LIMIT 5000
""").fetchall()
geojson_residential = json.dumps({
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lng, lat]},
            "properties": {}
        }
        for lng, lat in sample_residential
    ]
})

# check nonresidential, too
sample_nonresidential = conn.execute("""
    SELECT ST_X(geom) AS lng, ST_Y(geom) AS lat
    FROM addresses
    WHERE RESIDENTIA='NON RESIDENTIAL'
    ORDER BY RANDOM()
    LIMIT 5000
""").fetchall()

geojson_nonresidential = json.dumps({
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lng, lat]},
            "properties": {}
        }
        for lng, lat in sample_nonresidential
    ]
})

html_template = """<!DOCTYPE html>
<html>
<head>
  <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.css" rel="stylesheet">
  <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.js"></script>
  <style>html, body { margin: 0; height: 100%; } #map { width: 100%; height: 100%; }</style>
</head>
<body>
<div id="map"></div>
<script>
  mapboxgl.accessToken = 'ACCESS_TOKEN';
  const map = new mapboxgl.Map({
    container: 'map',
    style: 'mapbox://styles/mapbox/satellite-streets-v12',
    center: [-77.0369, 38.9072],
    zoom: 12
  });
  map.on('load', () => {
    map.addSource('addresses_residential_src', { type: 'geojson', data: GEOJSON_DATA_RESIDENTIAL });
    map.addLayer({
      id: 'addresses_residential_layer',
      type: 'circle',
      source: 'addresses_residential_src',
      paint: { 'circle-radius': 3, 'circle-color': 'red', 'circle-opacity': 1.0 }
    });

    map.addSource('addresses_nonresidential_src', { type: 'geojson', data: GEOJSON_DATA_NONRESIDENTIAL });
    map.addLayer({
      id: 'addresses_nonresidential_layer',
      type: 'circle',
      source: 'addresses_nonresidential_src',
      paint: { 'circle-radius': 3, 'circle-color': 'cyan', 'circle-opacity': 1.0 }
    });
  });
</script>
</body>
</html>"""


html = html_template.replace('ACCESS_TOKEN', MAPBOX_ACCESS_TOKEN)
html = html.replace('GEOJSON_DATA_RESIDENTIAL', geojson_residential)
html = html.replace('GEOJSON_DATA_NONRESIDENTIAL', geojson_nonresidential)
with open('test.html', 'w') as f:
    f.write(html)
IFrame(src=f"data:text/html;base64,{base64.b64encode(html.encode()).decode()}", width="100%", height="480px")

### 4. Visualize Schools

In [ ]:
rows = conn.execute("""
    SELECT NAME, ST_X(geom) AS lng, ST_Y(geom) AS lat
    FROM schools
""").fetchall()

geojson = json.dumps({
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lng, lat]},
            "properties": {"name": name}
        }
        for name, lng, lat in rows
    ]
})

html_template = """<!DOCTYPE html>
<html>
<head>
  <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.css" rel="stylesheet">
  <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.js"></script>
  <style>html, body { margin: 0; height: 100%; } #map { width: 100%; height: 100%; }</style>
</head>
<body>
<div id="map"></div>
<script>
  mapboxgl.accessToken = 'ACCESS_TOKEN';
  const map = new mapboxgl.Map({
    container: 'map',
    style: 'mapbox://styles/mapbox/light-v11',
    center: [-77.0369, 38.9072],
    zoom: 11
  });
  map.on('load', () => {
    map.addSource('schools', { type: 'geojson', data: GEOJSON_DATA });
    map.addLayer({
      id: 'schools',
      type: 'circle',
      source: 'schools',
      paint: { 'circle-radius': 5, 'circle-color': '#3887be', 'circle-stroke-width': 1, 'circle-stroke-color': '#fff' }
    });
    map.on('click', 'schools', (e) => {
      new mapboxgl.Popup().setLngLat(e.lngLat).setHTML(e.features[0].properties.name).addTo(map);
    });
    map.on('mouseenter', 'schools', () => { map.getCanvas().style.cursor = 'pointer'; });
    map.on('mouseleave', 'schools', () => { map.getCanvas().style.cursor = ''; });
  });
</script>
</body>
</html>"""

html = html_template.replace('ACCESS_TOKEN', MAPBOX_ACCESS_TOKEN).replace('GEOJSON_DATA', geojson)
IFrame(src=f"data:text/html;base64,{base64.b64encode(html.encode()).decode()}", width="100%", height="480px")

# 5. Generate Isochrones

An **isochrone** is a polygon representing everything reachable from a point within a given travel time. You can experiment with creating them on the [Mapbox Isochrone Playground](https://docs.mapbox.com/playground/isochrone/).

We'll call the [Mapbox Isochrone API](https://docs.mapbox.com/api/navigation/isochrone/) to get 10-minute walking polygons for each elementary school.

The API endpoint format is:
```
GET https://api.mapbox.com/isochrone/v1/mapbox/walking/{lng},{lat}
    ?contours_minutes=10&polygons=true&access_token={token}
```

We'll save each response as a GeoJSON file in the `isochrones/` folder. If you run this cell again, it skips schools that already have a file (so you can resume if interrupted).

In [ ]:
import os
import re
import json
import time
import requests

os.makedirs("isochrones", exist_ok=True)

def safe_filename(name):
    """Convert a school name to a safe filename."""
    return re.sub(r'[^\w]', '_', name)

# Filter to elementary schools — adjust the value if LEVEL_ uses a different label
elementary_schools = conn.execute("""
    SELECT NAME, ST_X(geom) AS lng, ST_Y(geom) AS lat
    FROM schools
    WHERE LEVEL_ = 'ES'
""").fetchall()

print(f"Found {len(elementary_schools)} elementary schools")

for name, lng, lat in elementary_schools:
    filepath = f"isochrones/{safe_filename(name)}.geojson"

    if os.path.exists(filepath):
        print(f"  skip  {name}")
        continue

    url = (
        f"https://api.mapbox.com/isochrone/v1/mapbox/walking/{lng},{lat}"
        f"?contours_minutes=10&polygons=true&access_token={MAPBOX_ACCESS_TOKEN}"
    )
    response = requests.get(url)
    response.raise_for_status()

    data = response.json()
    data["school_name"] = name  # tag the file with the school name for easy reload

    with open(filepath, "w") as f:
        json.dump(data, f)

    print(f"  saved {name}")
    time.sleep(0.1)  # stay well within API rate limits

print("Done!")

## 7. Load Isochrones into DuckDB

We'll add an `isochrone` geometry column to the `schools` table and populate it from the GeoJSON files.

In [ ]:
import glob

conn.execute("ALTER TABLE schools ADD COLUMN IF NOT EXISTS isochrone GEOMETRY")

for filepath in glob.glob("isochrones/*.geojson"):
    with open(filepath) as f:
        data = json.load(f)

    school_name = data.get("school_name")
    if not school_name:
        continue

    # GeoJSON features wrap geometries, and allow them to be paired with extra data ("properties").
    # Here, we just want the geometry from each isochrone feature.
    geometry = json.dumps(data["features"][0]["geometry"])

    conn.execute(
        "UPDATE schools SET isochrone = ST_GeomFromGeoJSON(?) WHERE NAME = ?",
        [geometry, school_name]
    )

loaded = conn.execute("SELECT COUNT(*) FROM schools WHERE isochrone IS NOT NULL").fetchone()[0]
print(f"Loaded isochrones for {loaded} schools")

## 8. Visualize Schools with Isochrones

In [ ]:
rows = conn.execute("""
    SELECT NAME, ST_X(geom) AS lng, ST_Y(geom) AS lat, ST_AsGeoJSON(isochrone) AS iso
    FROM schools
    WHERE isochrone IS NOT NULL
""").fetchall()

features = []
for name, lng, lat, iso in rows:
    features.append({
        "type": "Feature",
        "geometry": json.loads(iso),
        "properties": {"name": name}
    })
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lng, lat]},
        "properties": {"name": name}
    })

geojson = json.dumps({"type": "FeatureCollection", "features": features})

html_template = """<!DOCTYPE html>
<html>
<head>
  <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.css" rel="stylesheet">
  <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.js"></script>
  <style>html, body { margin: 0; height: 100%; } #map { width: 100%; height: 100%; }</style>
</head>
<body>
<div id="map"></div>
<script>
  mapboxgl.accessToken = 'ACCESS_TOKEN';
  const map = new mapboxgl.Map({
    container: 'map',
    style: 'mapbox://styles/mapbox/light-v11',
    center: [-77.0369, 38.9072],
    zoom: 11
  });
  map.on('load', () => {
    map.addSource('data', { type: 'geojson', data: GEOJSON_DATA });
    map.addLayer({
      id: 'isochrones',
      type: 'fill',
      source: 'data',
      filter: ['==', ['geometry-type'], 'Polygon'],
      paint: { 'fill-color': '#3887be', 'fill-opacity': 0.15 }
    });
    map.addLayer({
      id: 'isochrones-outline',
      type: 'line',
      source: 'data',
      filter: ['==', ['geometry-type'], 'Polygon'],
      paint: { 'line-color': '#3887be', 'line-width': 1.5 }
    });
    map.addLayer({
      id: 'schools',
      type: 'circle',
      source: 'data',
      filter: ['==', ['geometry-type'], 'Point'],
      paint: { 'circle-radius': 5, 'circle-color': '#e55e5e', 'circle-stroke-width': 1, 'circle-stroke-color': '#fff' }
    });
    map.on('click', 'schools', (e) => {
      new mapboxgl.Popup().setLngLat(e.lngLat).setHTML(e.features[0].properties.name).addTo(map);
    });
    map.on('mouseenter', 'schools', () => { map.getCanvas().style.cursor = 'pointer'; });
    map.on('mouseleave', 'schools', () => { map.getCanvas().style.cursor = ''; });
  });
</script>
</body>
</html>"""

html = html_template.replace('ACCESS_TOKEN', MAPBOX_ACCESS_TOKEN).replace('GEOJSON_DATA', geojson)
IFrame(src=f"data:text/html;base64,{base64.b64encode(html.encode()).decode()}", width="100%", height="480px")

## 9. Walkability Analysis

Now we run a **spatial join**: for each address point, we check whether it falls inside any school's isochrone polygon. DuckDB evaluates this with `ST_Within` and groups the results by school.

In [ ]:
conn.execute("""
    CREATE OR REPLACE TABLE walkability AS
    SELECT
        schools.NAME AS school_name,
        COUNT(addresses.geom) AS address_count
    FROM   schools
    JOIN   addresses ON ST_Within(addresses.geom, schools.isochrone)
    WHERE addresses.RESIDENTIA!='NON RESIDENTIAL'
    GROUP  BY schools.NAME
    ORDER  BY address_count DESC
""")

for row in conn.execute("SELECT school_name, address_count FROM walkability").fetchall():
    print(f"{row[1]:>6}  {row[0]}")

## Interactive Map

Each school is shown as a circle. **Larger circles = more addresses within a 10-minute walk.** Click any circle to see the school name and address count.

In [ ]:
rows = conn.execute("""
    SELECT walkability.school_name, ST_X(schools.geom) AS lng, ST_Y(schools.geom) AS lat, walkability.address_count
    FROM walkability
    JOIN schools ON walkability.school_name = schools.SCHOOL_NAM
""").fetchall()

geojson = json.dumps({
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lng, lat]},
            "properties": {"name": name, "address_count": count}
        }
        for name, lng, lat, count in rows
    ]
})

html_template = """<!DOCTYPE html>
<html>
<head>
  <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.css" rel="stylesheet">
  <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.0/mapbox-gl.js"></script>
  <style>html, body { margin: 0; height: 100%; } #map { width: 100%; height: 100%; }</style>
</head>
<body>
<div id="map"></div>
<script>
  mapboxgl.accessToken = 'ACCESS_TOKEN';
  const map = new mapboxgl.Map({
    container: 'map',
    style: 'mapbox://styles/mapbox/light-v11',
    center: [-77.0369, 38.9072],
    zoom: 11
  });
  map.on('load', () => {
    map.addSource('schools', { type: 'geojson', data: GEOJSON_DATA });
    map.addLayer({
      id: 'schools',
      type: 'circle',
      source: 'schools',
      paint: {
        'circle-radius': ['interpolate', ['linear'], ['get', 'address_count'], 0, 4, 1000, 22],
        'circle-color': '#3887be',
        'circle-opacity': 0.85,
        'circle-stroke-width': 1.5,
        'circle-stroke-color': '#ffffff'
      }
    });
    map.on('click', 'schools', (e) => {
      const p = e.features[0].properties;
      new mapboxgl.Popup()
        .setLngLat(e.lngLat)
        .setHTML('<strong>' + p.name + '</strong><br>' + p.address_count + ' addresses within 10-min walk')
        .addTo(map);
    });
    map.on('mouseenter', 'schools', () => { map.getCanvas().style.cursor = 'pointer'; });
    map.on('mouseleave', 'schools', () => { map.getCanvas().style.cursor = ''; });
  });
</script>
</body>
</html>"""

html = html_template.replace('ACCESS_TOKEN', MAPBOX_ACCESS_TOKEN).replace('GEOJSON_DATA', geojson)
IFrame(src=f"data:text/html;base64,{base64.b64encode(html.encode()).decode()}", width="100%", height="520px")

## Walkability vs. Enrollment

Does a school's walkability correlate with how many students it enrolls? Let's plot the number of residential addresses within each school's isochrone against total student enrollment.

In [ ]:
import matplotlib.pyplot as plt

rows = conn.execute("""
    SELECT w.address_count, s.TOTAL_STUD, w.school_name
    FROM walkability w
    JOIN schools s ON w.school_name = s.SCHOOL_NAM
    WHERE s.TOTAL_STUD IS NOT NULL
""").fetchall()

address_counts = [r[0] for r in rows]
enrollments    = [r[1] for r in rows]
names          = [r[2] for r in rows]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(enrollments, address_counts, alpha=0.7, edgecolors='white', linewidths=0.5, s=60)

for x, y, name in zip(enrollments, address_counts, names):
    ax.annotate(name, (x, y), fontsize=6, alpha=0.75,
                xytext=(4, 4), textcoords='offset points')

ax.set_xlabel("Total student enrollment")
ax.set_ylabel("Addresses within 10-min walking isochrone")
ax.set_title("School walkability vs. enrollment")

plt.tight_layout()
plt.show()

# Questions?
### tlee@mapbox.com

# Github
![qr code for repo](qr.png)